# IRIS eager FP32 CUDA qualification
Select **Runtime → Change runtime type → GPU** first, then run the cells in order in a fresh runtime.

This uses a separate Python 3.12 environment. It compares the official and M* networks at B=1 and B=2, crosses the context refresh, and exercises actual CUDA sampling and request departure/reuse. It does not run the API server, enable CUDA graphs, or measure performance.

The first kernel use can compile. For reruns, use a new output name in the execution/download cells, without rerunning the clone cell. Download the JSON report after either a pass or a failure. See `scripts/iris/CUDA.md` for details.


In [ ]:
%pip install -q uv

In [ ]:
%%bash
set -euo pipefail
git clone --branch codex/iris --single-branch https://github.com/Artifex2002/mstar.git /content/mstar-iris
cd /content/mstar-iris
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python --torch-backend=auto -r scripts/iris/requirements-cuda-tests.txt
mkdir -p .ci-results/iris
git clone https://github.com/eloialonso/iris.git .ci-results/iris/reference
git -C .ci-results/iris/reference checkout --detach 24326aaaa283c527f42b89b44cfdecf2665a7a16


In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
OMP_NUM_THREADS=2 .venv/bin/python -m scripts.iris.qualify_cuda --download-checkpoint --output .ci-results/iris/cuda-check-01 2>&1 | tee .ci-results/iris/cuda-check-01.log


If the execution cell failed, still run the download cell. If installation failed before a report exists, send the cell output instead.


In [ ]:
from google.colab import files

files.download("/content/mstar-iris/.ci-results/iris/cuda-check-01/report.json")